# Benchmark Result Comparison

Use this notebook after running the pipeline on several synthetic benchmark datasets. It collects per-case generation quality, formula quality, prediction metrics, recovery metrics, homogeneity error, active-invariant recovery, and classification.

In [ ]:
from pathlib import Path
import sys
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from invariant_generator.benchmark import collect_benchmark_results
from invariant_generator.config import load_config

CONFIG_PATH = PROJECT_ROOT / 'configs/self_eval_benchmark.toml'
config = load_config(CONFIG_PATH)
comparison = collect_benchmark_results(config)
rows = comparison['rows']

print('comparison json:', comparison['summary_path'])
print('completed:', comparison['n_completed'], '/', comparison['n_cases'])
pprint(comparison['classification_counts'])

In [ ]:
for row in rows:
    print('\n', row['case_id'])
    print('  difficulty:', row['difficulty'])
    print('  classification:', row['classification'])
    print('  test_rmse:', row['test_rmse'])
    print('  recovery_relative_l2:', row['recovery_relative_l2'])
    print('  homogeneity_error:', row['homogeneity_error'])
    print('  active_invariant_match:', row['active_invariant_match'])
    print('  best_equation:', row['best_equation'])

In [ ]:
labels = [row['case_id'] for row in rows]
rel_l2 = np.array([
    np.nan if row['recovery_relative_l2'] is None else row['recovery_relative_l2']
    for row in rows
], dtype=float)
hom = np.array([
    np.nan if row['homogeneity_error'] is None else row['homogeneity_error']
    for row in rows
], dtype=float)
test_rmse = np.array([
    np.nan if row['test_rmse'] is None else row['test_rmse']
    for row in rows
], dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].bar(labels, rel_l2)
axes[0].set_yscale('log')
axes[0].set_title('Formula recovery relative L2')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(labels, hom)
axes[1].set_yscale('log')
axes[1].set_title('Recovered formula homogeneity')
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(labels, test_rmse)
axes[2].set_yscale('log')
axes[2].set_title('Pipeline test RMSE')
axes[2].tick_params(axis='x', rotation=45)

fig.tight_layout()

In [ ]:
counts = comparison['classification_counts']
if counts:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(list(counts), list(counts.values()))
    ax.set_title('Recovery classification counts')
    ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()
else:
    print('No completed recovery metrics found yet.')